In [27]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.utils.class_weight import compute_class_weight
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline as SkPipeline

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neural_network import MLPClassifier

from sklearn.naive_bayes import GaussianNB


### 0. Get processed splits

In [28]:
SPLIT_DIR = Path("../Data/splits")
TARGET = "TARGET_STAFF_CUT"

# Load processed splits
train_proc = pd.read_csv(SPLIT_DIR / "train_processed.csv")
valid_proc = pd.read_csv(SPLIT_DIR / "valid_processed.csv")
test_proc  = pd.read_csv(SPLIT_DIR / "test_processed.csv")

# Extract X and y
X_train = train_proc.drop(columns=[TARGET]).values
y_train = train_proc[TARGET].values

X_valid = valid_proc.drop(columns=[TARGET]).values
y_valid = valid_proc[TARGET].values

X_test = test_proc.drop(columns=[TARGET]).values
y_test = test_proc[TARGET].values

Training data contains years 2014-2022. \
Validation data contains year 2023.  \
Test data contains year 2024

### 1. Train and evaluate Linear and Tree Models

In [29]:
warnings.filterwarnings("ignore")

# Class Imbalance Handling
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
cw_dict = {0: class_weights[0], 1: class_weights[1]}

# Initial Model Definitions
base_models = {
    "Logistic": LogisticRegression(max_iter=5000, class_weight="balanced"),

    "RandomForest": RandomForestClassifier(
        class_weight="balanced", random_state=42
    ),

    "ExtraTrees": ExtraTreesClassifier(
        class_weight="balanced", random_state=42
    ),

    "HistGB": HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_depth=6,
        max_leaf_nodes=31,
        class_weight="balanced"
    ),

    "XGBoost": xgb.XGBClassifier(
        scale_pos_weight=pos_weight, eval_metric="aucpr", random_state=42
    ),

    "LightGBM": lgb.LGBMClassifier(
        scale_pos_weight=pos_weight, random_state=42, verbose=-1
    ),

    "CatBoost": CatBoostClassifier(
        iterations=400,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_state=42,
        verbose=False,
        auto_class_weights="Balanced"   # FIXED
    ),

    "LinearSVM": CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", C=1.0),
        method="sigmoid",
        cv=3
    ),

    "RBFSVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        C=1.0,
        gamma="scale"
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        max_iter=500,
        alpha=0.0005
    ),

    "NaiveBayes": GaussianNB()
}

# Hyperparameter grids for RandomizedSearchCV
param_grids = {
    "Logistic": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs", "liblinear"]
    },

    "RandomForest": {
        "n_estimators": [200, 500, 800],
        "max_depth": [6, 8, 12],
        "min_samples_leaf": [1, 5, 20],
        "max_features": ["sqrt", "log2"]
    },

    "ExtraTrees": {
        "n_estimators": [200, 500, 800],
        "max_depth": [None, 8, 12],
        "min_samples_leaf": [1, 5, 20],
        "max_features": ["sqrt", "log2"]
    },

    "HistGB": {
        "learning_rate": [0.01, 0.03, 0.1],
        "max_depth": [4, 6, 8],
        "max_leaf_nodes": [31, 50, 100]
    },

    "XGBoost": {
        "n_estimators": [200, 400, 600],
        "learning_rate": [0.01, 0.03, 0.1],
        "max_depth": [3, 4, 6],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.7, 1.0]
    },

    "LightGBM": {
        "n_estimators": [200, 400, 600],
        "learning_rate": [0.01, 0.03, 0.1],
        "num_leaves": [31, 50, 100],
        "min_child_samples": [20, 40, 60],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.7, 1.0]
    },

    "CatBoost": {
        "iterations": [200, 400, 600],
        "learning_rate": [0.01, 0.03, 0.1],
        "depth": [4, 6, 8]
    },

    "LinearSVM": {
        "estimator__C": [0.01, 0.1, 1, 10]   # FIXED
    },

    "RBFSVM": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", "auto"]
    },

    "MLP": {
        "hidden_layer_sizes": [(64,), (128, 64), (256, 128)],
        "alpha": [0.0001, 0.0005, 0.001],
        "learning_rate_init": [0.001, 0.01]
    },

    "NaiveBayes": {}
}

# Train and evaluate each model with hyperparameter tuning
rows = []

for name, model in base_models.items():

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids[name],
        n_iter=10,
        scoring="average_precision",
        cv=3,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    # Evaluate on train/valid/test
    for split, X_split, y_split in [
        ("train", X_train, y_train),
        ("validation", X_valid, y_valid),
        ("test", X_test, y_test)
    ]:
        proba = best_model.predict_proba(X_split)[:, 1]
        base = y_split.mean()
        prc = average_precision_score(y_split, proba)

        rows.append({
            "model": name,
            "split": split,
            "ROC_AUC": round(roc_auc_score(y_split, proba), 3),
            "PR_AUC": round(prc, 3),
            "base_rate": round(base, 3),
            "PR_lift": round(prc / base, 2),
            "best_params": search.best_params_
        })

comparison = pd.DataFrame(rows)
print(comparison[comparison["split"] == "test"].to_string(index=False))

/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fit

       model split  ROC_AUC  PR_AUC  base_rate  PR_lift                                                                                                                        best_params
    Logistic  test    0.637   0.253      0.181     1.39                                                                                                 {'solver': 'liblinear', 'C': 0.01}
RandomForest  test    0.699   0.339      0.181     1.87                                             {'n_estimators': 800, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 12}
  ExtraTrees  test    0.623   0.249      0.181     1.37                                             {'n_estimators': 500, 'min_samples_leaf': 20, 'max_features': 'log2', 'max_depth': 12}
      HistGB  test    0.669   0.290      0.181     1.60                                                                      {'max_leaf_nodes': 31, 'max_depth': 4, 'learning_rate': 0.01}
     XGBoost  test    0.692   0.337      0.181     1.86          

**Takeaways:** Hypertuned tree-based models (RandomForest, XGBoost and LightGBM) are the top performers in terms of PR-AUC and PR-Lift. Based on the results, RandomForest has the best overall performance, stable with a high bias and stability and 1.87 PR_lift with means that it predicts 1.87 better compared to a random model. XGBoost nearly tied with RandomForest as it captured non linear structure. Perhaps, a stacked ensemble of the top 3 could puush PR-AUC higher.

In [30]:

# Get top 3 models based on PR_AUC from the tree models (RandomForest, XGBoost, LightGBM)
best_rf      = comparison[(comparison.model == "RandomForest")].iloc[0]["best_params"]
best_xgb     = comparison[(comparison.model == "XGBoost")].iloc[0]["best_params"]
best_lgbm    = comparison[(comparison.model == "LightGBM")].iloc[0]["best_params"]

# Re‑instantiate models with best params
rf_model = RandomForestClassifier(
    **best_rf,
    class_weight="balanced",
    random_state=42
)

xgb_model = xgb.XGBClassifier(
    **best_xgb,
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",
    random_state=42
)

lgbm_model = lgb.LGBMClassifier(
    **best_lgbm,
    scale_pos_weight=pos_weight,
    random_state=42,
    verbose=-1
)

# Meta‑learner (simple, stable)
meta_model = LogisticRegression(
    solver="liblinear",
    C=0.01,
    max_iter=500,
    class_weight="balanced"
)


# Stacking Ensemble
from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model),
        ('lgbm', lgbm_model)
    ],
    final_estimator=meta_model,
    stack_method='predict_proba',
    n_jobs=-1
)

# Train ensemble
stack.fit(X_train, y_train)

# Evaluate on test
stack_proba = stack.predict_proba(X_test)[:, 1]

stack_roc = roc_auc_score(y_test, stack_proba)
stack_pr  = average_precision_score(y_test, stack_proba)

print("STACKED ENSEMBLE RESULTS")
print("ROC_AUC:", round(stack_roc, 3))
print("PR_AUC :", round(stack_pr, 3))
print("PR_LIFT:", round(stack_pr / y_test.mean(), 2))


/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fit

STACKED ENSEMBLE RESULTS
ROC_AUC: 0.698
PR_AUC : 0.348
PR_LIFT: 1.92


**Takeaway:** Ensemble method with top 3 performers outperform the best individual models based on PR-AUC and the lift increased to 1.92 as well.

### 3. Resampling using SMOTE

In [31]:
# SMOTE + Tomek + Model Pipeline
# Resampling inside each CV fold to avoid data leakage
smote_pipeline = Pipeline([
    ("smote", SMOTETomek(sampling_strategy="auto")),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

# Hyperparameter grid for RandomizedSearchCV
param_grid = {
    "model__n_estimators": [200, 500, 800],
    "model__max_depth": [6, 8, 12],
    "model__min_samples_leaf": [1, 5, 20],
    "model__max_features": ["sqrt", "log2"]
}

search = RandomizedSearchCV(
    estimator=smote_pipeline,
    param_distributions=param_grid,
    n_iter=10,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

# Training
search.fit(X_train, y_train)
best_model = search.best_estimator_

rows = []

# Evaluate on train/valid/test
for split, X_split, y_split in [
    ("train", X_train, y_train),
    ("validation", X_valid, y_valid),
    ("test", X_test, y_test)
]:
    proba = best_model.predict_proba(X_split)[:, 1]
    base = y_split.mean()
    prc = average_precision_score(y_split, proba)

    rows.append({
        "split": split,
        "ROC_AUC": round(roc_auc_score(y_split, proba), 3),
        "PR_AUC": round(prc, 3),
        "base_rate": round(base, 3),
        "PR_lift": round(prc / base, 2),
        "best_params": search.best_params_
    })

results = pd.DataFrame(rows)
print(results[results["split"] == "test"].to_string(index=False))

/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/jasonavalos/Projects/school/ADS599/Capstone/academic-staff-safeguard/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validatio

split  ROC_AUC  PR_AUC  base_rate  PR_lift                                                                                                      best_params
 test    0.685    0.31      0.181     1.71 {'model__n_estimators': 200, 'model__min_samples_leaf': 5, 'model__max_features': 'sqrt', 'model__max_depth': 6}


**Takeway:** Resampling using SMOTE did not improve the model performance based on AUC as the value decrease to .692 and lift decreased as well to 1.77

### Training Linear Models with PCA

In [32]:
# Linear Models Setup
linear_models = {
    "Logistic": LogisticRegression(max_iter=5000, class_weight="balanced"),
    "LinearSVM": CalibratedClassifierCV(
        LinearSVC(class_weight="balanced"),
        method="sigmoid",
        cv=3
    ),
    "RBFSVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced"
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        max_iter=500
    )
}


# Hyperparameter grids for RandomizedSearchCV with PCA
param_grids = {
    "Logistic": {
        "model__C": [0.01, 0.1, 1, 10]
    },
    "LinearSVM": {
        "model__estimator__C": [0.01, 0.1, 1, 10]
    },
    "RBFSVM": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", "auto"]
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64,), (128, 64), (256, 128)],
        "model__alpha": [0.0001, 0.0005, 0.001],
        "model__learning_rate_init": [0.001, 0.01]
    }
}

# Train + evaluate PCA pipelines
rows = []

for name, model in linear_models.items():

    pca_pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.95)),
        ("model", model)
    ])

    search = RandomizedSearchCV(
        estimator=pca_pipeline,
        param_distributions=param_grids[name],
        n_iter=10,
        scoring="average_precision",
        cv=3,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    for split, X_split, y_split in [
        ("train", X_train, y_train),
        ("validation", X_valid, y_valid),
        ("test", X_test, y_test)
    ]:
        proba = best_model.predict_proba(X_split)[:, 1]
        base = y_split.mean()
        prc = average_precision_score(y_split, proba)

        rows.append({
            "model": name,
            "split": split,
            "ROC_AUC": round(roc_auc_score(y_split, proba), 3),
            "PR_AUC": round(prc, 3),
            "base_rate": round(base, 3),
            "PR_lift": round(prc / base, 2),
            "best_params": search.best_params_
        })

results = pd.DataFrame(rows)
print(results[results["split"] == "test"].to_string(index=False))

    model split  ROC_AUC  PR_AUC  base_rate  PR_lift                                                                                         best_params
 Logistic  test    0.624   0.241      0.181     1.33                                                                                  {'model__C': 0.01}
LinearSVM  test    0.600   0.226      0.181     1.25                                                                       {'model__estimator__C': 0.01}
   RBFSVM  test    0.622   0.259      0.181     1.43                                                          {'model__gamma': 'scale', 'model__C': 0.1}
      MLP  test    0.558   0.236      0.181     1.30 {'model__learning_rate_init': 0.01, 'model__hidden_layer_sizes': (128, 64), 'model__alpha': 0.0001}


**Takeway:** PCA did not contribute to any improvement to the performance metrics.